Khởi động Spark

In [1]:
import os
import sys
from pyspark.sql import SparkSession
import pandas as pd
from datasets import load_dataset

# Thuốc giải lỗi đường dẫn Python trên Windows
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

print("--- KHỞI ĐỘNG HỆ THỐNG ETL ---")

# Khởi tạo Spark (Ép dùng tối đa 2GB RAM để bảo vệ máy)
spark = SparkSession.builder \
    .appName("HM_Data_Pipeline") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()
print("✅ Spark đã sẵn sàng! Version:", spark.version)

c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- KHỞI ĐỘNG HỆ THỐNG ETL ---
✅ Spark đã sẵn sàng! Version: 4.1.1


Extract - Kéo dữ liệu

In [2]:
print("⏳ Đang tải dữ liệu H&M từ HuggingFace (định dạng Arrow/Parquet tối ưu)...")

# Tải bộ dataset
dataset = load_dataset("Qdrant/hm_ecommerce_products", split="train")

print(f"✅ Đã tải xong! Tổng số dòng dữ liệu gốc: {len(dataset)}")
# Xem thử 1 dòng dữ liệu đầu tiên trông như thế nào
dataset[0]

⏳ Đang tải dữ liệu H&M từ HuggingFace (định dạng Arrow/Parquet tối ưu)...
✅ Đã tải xong! Tổng số dòng dữ liệu gốc: 105126


{'article_id': '0108775015',
 'product_code': '0108775',
 'prod_name': 'Strap top',
 'product_type_no': 253,
 'product_type_name': 'Vest top',
 'product_group_name': 'Garment Upper body',
 'graphical_appearance_no': 1010016,
 'graphical_appearance_name': 'Solid',
 'colour_group_code': '09',
 'colour_group_name': 'Black',
 'perceived_colour_value_id': 4,
 'perceived_colour_value_name': 'Dark',
 'perceived_colour_master_id': 5,
 'perceived_colour_master_name': 'Black',
 'department_no': 1676,
 'department_name': 'Jersey Basic',
 'index_code': 'A',
 'index_name': 'Ladieswear',
 'index_group_no': 1,
 'index_group_name': 'Ladieswear',
 'section_no': 16,
 'section_name': 'Womens Everyday Basics',
 'garment_group_no': 1002,
 'garment_group_name': 'Jersey Basic',
 'detail_desc': 'Jersey top with narrow shoulder straps.',
 'text_to_embed': 'Strap top Vest top Garment Upper body Solid Black Dark Black Jersey Basic Ladieswear Ladieswear Womens Everyday Basics Jersey Basic Jersey top with narrow s

Transform - Đưa vào Spark

In [3]:
import numpy as np

print("⏳ Đang xử lý dữ liệu từ Pandas...")
pdf = dataset.to_pandas()

# 2. BƯỚC LỌC DỮ LIỆU (Giữ lại đúng 6 cột quan trọng nhất theo tên gốc của dataset)
columns_to_keep = [
    'article_id',       # ID sản phẩm
    'prod_name',        # Tên sản phẩm
    'detail_desc',      # Mô tả chi tiết
    'department_name',  # Tên phòng ban/danh mục
    'image_url',        # Link ảnh
    'dense_embedding'   # Vector 384 chiều
]

# Chỉ lấy những cột này để siêu tiết kiệm RAM
pdf = pdf[columns_to_keep]

print("⏳ Đang ép kiểu cột Vector để PySpark đọc được...")
# 3. Chuyển đổi ndarray thành dạng List cơ bản
pdf['dense_embedding'] = pdf['dense_embedding'].apply(lambda x: x.tolist() if hasattr(x, 'tolist') else x)

print("⏳ Đang chuyển đổi sang Spark DataFrame...")
# 4. Đưa vào Spark
df_spark = spark.createDataFrame(pdf)

print("✅ Đã tạo Spark DataFrame thành công! Cấu trúc dữ liệu CHUẨN như sau:")
df_spark.printSchema()

# In thử 3 dòng đầu tiên xem mặt mũi dữ liệu ra sao nhé!
df_spark.show(3, truncate=True)

⏳ Đang xử lý dữ liệu từ Pandas...
⏳ Đang ép kiểu cột Vector để PySpark đọc được...
⏳ Đang chuyển đổi sang Spark DataFrame...


c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:416: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Cannot convert pyarrow.lib.ChunkedArray to pyarrow.lib.Array
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


✅ Đã tạo Spark DataFrame thành công! Cấu trúc dữ liệu CHUẨN như sau:
root
 |-- article_id: string (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- detail_desc: string (nullable = true)
 |-- department_name: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- dense_embedding: array (nullable = true)
 |    |-- element: double (containsNull = true)

+----------+-------------+--------------------+---------------+--------------------+--------------------+
|article_id|    prod_name|         detail_desc|department_name|           image_url|     dense_embedding|
+----------+-------------+--------------------+---------------+--------------------+--------------------+
|0108775015|    Strap top|Jersey top with n...|   Jersey Basic|https://qdrant-ne...|[-0.0400244370102...|
|0108775044|    Strap top|Jersey top with n...|   Jersey Basic|https://qdrant-ne...|[-0.0565592609345...|
|0108775051|Strap top (1)|Jersey top with n...|   Jersey Basic|https://qdrant-ne...|[-0

In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct

print("⏳ Đang kết nối với Qdrant...")
client = QdrantClient(url="http://localhost:6333", timeout=60)
collection_name = "Qdrant_Group10"

print("🚀 Đang BƠM DỮ LIỆU bằng Python Native (Bypass lỗi PySpark trên Windows)...")

batch_size = 1000  # Trả về mức 1.000 dòng để Qdrant không bị nghẹn
points = []
batch_count = 1
total_inserted = 0

# Dùng biến pdf (Pandas) thay vì df_spark để tránh lỗi sập nguồn của Py4J
for index, row in pdf.iterrows():
    # 1. Đóng gói Hành lý
    payload = {
        "product_name": row['prod_name'],
        "description": row['detail_desc'],
        "department_name": row['department_name'],
        "image_url": row['image_url']
    }
    
    # 2. Tạo Point
    point = PointStruct(
        id=int(row['article_id']),
        vector=row['dense_embedding'],
        payload=payload
    )
    points.append(point)
    
    # 3. Đủ 1000 dòng thì đẩy
    if len(points) >= batch_size:
        try:
            client.upsert(collection_name=collection_name, points=points)
            total_inserted += len(points)
            print(f"✅ Đã đẩy mẻ {batch_count} (Tổng: {total_inserted} sản phẩm)...")
            points = [] # Xóa hộp dọn RAM
            batch_count += 1
        except Exception as e:
            print(f"❌ Lỗi ở mẻ {batch_count}: {e}")

# Đẩy nốt phần lẻ cuối cùng
if len(points) > 0:
    client.upsert(collection_name=collection_name, points=points)
    total_inserted += len(points)

print(f"\n🎉 XUẤT SẮC! Đã bơm THÀNH CÔNG {total_inserted} sản phẩm vào Qdrant!")

⏳ Đang kết nối với Qdrant...
🚀 Đang BƠM DỮ LIỆU bằng Python Native (Bypass lỗi PySpark trên Windows)...
✅ Đã đẩy mẻ 1 (Tổng: 1000 sản phẩm)...
✅ Đã đẩy mẻ 2 (Tổng: 2000 sản phẩm)...
✅ Đã đẩy mẻ 3 (Tổng: 3000 sản phẩm)...
✅ Đã đẩy mẻ 4 (Tổng: 4000 sản phẩm)...
✅ Đã đẩy mẻ 5 (Tổng: 5000 sản phẩm)...
✅ Đã đẩy mẻ 6 (Tổng: 6000 sản phẩm)...
✅ Đã đẩy mẻ 7 (Tổng: 7000 sản phẩm)...
✅ Đã đẩy mẻ 8 (Tổng: 8000 sản phẩm)...
✅ Đã đẩy mẻ 9 (Tổng: 9000 sản phẩm)...
✅ Đã đẩy mẻ 10 (Tổng: 10000 sản phẩm)...
✅ Đã đẩy mẻ 11 (Tổng: 11000 sản phẩm)...
✅ Đã đẩy mẻ 12 (Tổng: 12000 sản phẩm)...
✅ Đã đẩy mẻ 13 (Tổng: 13000 sản phẩm)...
✅ Đã đẩy mẻ 14 (Tổng: 14000 sản phẩm)...
✅ Đã đẩy mẻ 15 (Tổng: 15000 sản phẩm)...
✅ Đã đẩy mẻ 16 (Tổng: 16000 sản phẩm)...
✅ Đã đẩy mẻ 17 (Tổng: 17000 sản phẩm)...
✅ Đã đẩy mẻ 18 (Tổng: 18000 sản phẩm)...
✅ Đã đẩy mẻ 19 (Tổng: 19000 sản phẩm)...
✅ Đã đẩy mẻ 20 (Tổng: 20000 sản phẩm)...
✅ Đã đẩy mẻ 21 (Tổng: 21000 sản phẩm)...
✅ Đã đẩy mẻ 22 (Tổng: 22000 sản phẩm)...
✅ Đã đẩy mẻ 

In [5]:
print("🔍 Đang KIỂM TRA tính năng tìm kiếm tương đồng của Qdrant...")

# 1. Lấy thử vector của sản phẩm đầu tiên
test_vector = pdf['dense_embedding'].iloc[0]
test_name = pdf['prod_name'].iloc[0]

print(f"👉 Đang tìm các sản phẩm giống với: '{test_name}'...")

# 2. Sử dụng hàm query_points (Thay cho search để tránh lỗi AttributeError)
search_results = client.query_points(
    collection_name=collection_name,
    query=test_vector,
    limit=3
).points

# 3. In kết quả ra màn hình
print("\n✅ KẾT QUẢ TÌM KIẾM TỪ KHO DỮ LIỆU:")
print("-" * 50)
for i, hit in enumerate(search_results):
    print(f"--- Top {i+1} --- (Độ giống: {hit.score:.4f})")
    print(f"🔹 ID: {hit.id}")
    print(f"🔹 Tên SP: {hit.payload.get('product_name', 'N/A')}")
    print(f"🔹 Phân loại: {hit.payload.get('department_name', 'N/A')}")
    print(f"🔹 Link ảnh: {hit.payload.get('image_url', 'N/A')}")
    print("-" * 50)

🔍 Đang KIỂM TRA tính năng tìm kiếm tương đồng của Qdrant...
👉 Đang tìm các sản phẩm giống với: 'Strap top'...

✅ KẾT QUẢ TÌM KIẾM TỪ KHO DỮ LIỆU:
--------------------------------------------------
--- Top 1 --- (Độ giống: 1.0000)
🔹 ID: 108775015
🔹 Tên SP: Strap top
🔹 Phân loại: Jersey Basic
🔹 Link ảnh: https://qdrant-nextjs-demo-product-images.s3.us-east-1.amazonaws.com/images/010/0108775015.jpg
--------------------------------------------------
--- Top 2 --- (Độ giống: 0.9736)
🔹 ID: 812371001
🔹 Tên SP: Strap top 3-pack
🔹 Phân loại: Jersey Basic
🔹 Link ảnh: https://qdrant-nextjs-demo-product-images.s3.us-east-1.amazonaws.com/images/081/0812371001.jpg
--------------------------------------------------
--- Top 3 --- (Độ giống: 0.9676)
🔹 ID: 688463001
🔹 Tên SP: Straptop 2-pack
🔹 Phân loại: Jersey Basic
🔹 Link ảnh: https://qdrant-nextjs-demo-product-images.s3.us-east-1.amazonaws.com/images/068/0688463001.jpg
--------------------------------------------------
